<h3> Pandas Transformation of csv files

We are going to accomplish the same file preprocessing and transformations as the last homework, but use the pandas library instead.  Rather than having to "loop" through the data file, one line at a time, we will load the data file in its entirety into a dataFrame.  <p>Using Pandas results in more of a straight line scripting methodology that   resembles R and Rstudio.  The perspective is instead of treating each data line separately as in pure Python, the whole dataset is in a dataFrame and you can do the transformations to the columns directly rather than each item, one at a time.

In [124]:
# Name: Charley Kirk
# Date: 09/18/24
# Synopsis: this jupyter notebook creates transformed copies of weather.csv and SuicideChina.csv called weatherbinarized.csv and SuicideChinaTransf.csv respectively.

# set up for the pandas library
import csv
import pandas as pd
from pandas import Series, DataFrame

# input the file contents into a dataFrame.
weatherDF = pd.read_csv("/Users/Charley/Dropbox/Charley/Juniata/Data Science Masters/Data Acquisition & Visualization/weather.csv")
weatherDF

,Unnamed: 0,city,date,year,month,day,high_temp,avg_temp,low_temp,high_dewpt,...,avg_hg,low_hg,high_vis,avg_vis,low_vis,high_wind,avg_wind,low_wind,precip,events
0,1,Auckland,2016-01-01,2016,1,1,68,65,62,64,...,30.09,30.01,6,6,4,21,15,28.0,0,Rain
1,2,Auckland,2016-01-02,2016,1,2,68,66,64,64,...,29.90,29.80,6,5,1,33,21,46.0,0,Rain
2,3,Auckland,2016-01-03,2016,1,3,77,72,66,70,...,29.73,29.68,6,6,1,18,12,NaN,0,Rain
3,4,Auckland,2016-01-04,2016,1,4,73,66,60,66,...,29.90,29.77,6,6,6,15,10,NaN,0,Rain
4,5,Auckland,2016-01-05,2016,1,5,69,62,55,55,...,30.14,30.09,6,6,6,13,7,NaN,0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3650,3651,San Diego,2017-12-27,2017,12,27,69,60,50,56,...,30.15,30.09,10,5,1,13,4,16.0,0,NaN
3651,3652,San Diego,2017-12-28,2017,12,28,74,62,49,58,...,30.11,30.05,10,6,0,14,3,17.0,0,Fog
3652,3653,San Diego,2017-12-29,2017,12,29,77,63,49,59,...,30.11,30.06,10,5,0,8,2,10.0,0,Fog
3653,3654,San Diego,2017-12-30,2017,12,30,72,61,49,57,...,30.09,30.04,10,6,0,13,3,17.0,0,Fog


Follow the steps in the comments and code them using pandas dataFrame functions and operations.  In the last assignment using basic Python you thought in terms of the line of data and wrote code to alter the list of values in the line.  Here you think of the data in a table and manipulate the table columns in a single operation.
<br> Note that you can refer to the header name as the index of the column!

In [125]:
# delete the date string and id columns
del weatherDF['date']

drop = weatherDF.columns[weatherDF.columns.str.contains('Unnamed')]


weatherDF.drop(drop, axis=1, inplace=True)

# change the name of the 'low_wind' column to 'gust_wind'
weatherDF = weatherDF.rename(columns={'low_wind': 'gust_wind'})

# replace all instances of 'NA' in the 'gust_wind' column with the high_wind value

weatherDF['gust_wind']=weatherDF['gust_wind'].fillna(value=weatherDF['high_wind'])

# replace all 'T' instances in the 'precip' column with value 0.001
weatherDF['precip'] = [0.001 if 'T' in str(pre) else pre for pre in weatherDF['precip']]

# for each of the weather events in the 'events' column, add a new column with the event as its name
weatherDF['fog'] = [1 if 'Fog' in str(event) else 0 for event in weatherDF['events']]
weatherDF['rain'] = [1 if 'Rain' in str(event) else 0 for event in weatherDF['events']]
weatherDF['thunderstorm'] = [1 if 'Thunderstorm' in str(event) else 0 for event in weatherDF['events']]
weatherDF['snow'] = [1 if 'Snow' in str(event) else 0 for event in weatherDF['events']]
weatherDF['hail'] = [1 if 'Hail' in str(event) else 0 for event in weatherDF['events']]

# delete the 'events' column once you have binarized the events into separate columns
del weatherDF['events']

# finally you write out the dataFrame to a csv file in one step
weatherDF.to_csv("/Users/Charley/Dropbox/Charley/Juniata/Data Science Masters/Data Acquisition & Visualization/weatherbinarized.csv", header=True)

Now do similar steps for the China Suicide data. 
Copy and paste the code from above starting with the read_csv step.
<p>
    For purposes of verification of the chunking step below.  Alter this code to count the number of male and female data records.

In [126]:
# open and input the data into a dataFrame
sDF = pd.read_csv("/Users/Charley/Dropbox/Charley/Juniata/Data Science Masters/Data Acquisition & Visualization/SuicideChina.csv")


# eliminate the first two id columns
del sDF['Person_ID']


drop = sDF.columns[sDF.columns.str.contains('Unnamed')]


sDF.drop(drop, axis=1, inplace=True)

# The columns of Hospitalised, Died, Urban all have data coded as yes/no.  
#   Change the data to 1 or 0.  Anything not yes/no should be coded as NA.
for i in range(3):
    for k in range(2571):
        if sDF.iloc[k, i]=="yes":
            sDF.iloc[k, i]=1
        elif sDF.iloc[k, i]=="no":
            sDF.iloc[k, i]=0
        else:
            sDF.iloc[k, i]="NA"

# The data for Sex is male/female; recode as m/f.
def accessCharacter(x):
    return x[0]
sex=sDF["Sex"]
nsex=sex.apply(accessCharacter)
sDF["Sex"]=nsex

# Fix Education values to be all lower case. Replace 'unknown' values with NA.
def fixUp(x):
    x=x.lower()
    if x=='unknown':
        x='NA'
    return x
edu=sDF["Education"]
nedu=edu.apply(fixUp)
sDF["Education"]=nedu

# The data under method (which ought to be capitalized in the header), 
#     there are some 'unspecified' values, so replace them with NA
def fixUp2(x):
    if x=='unspecified':
        x='NA'
    return x
sDF = sDF.rename(columns={'method': 'Method'})
met=sDF["Method"]
nmet=met.apply(fixUp2)
sDF["Method"]=nmet


occ=sDF["Occupation"]
nocc=occ.apply(fixUp)
sDF["Occupation"]=nocc

# write the dataFrame out to csv file.
sDF.to_csv("/Users/Charley/Dropbox/Charley/Juniata/Data Science Masters/Data Acquisition & Visualization/SuicideChinaTransf.csv", header=True)

# calculate the number of male records and female records just as a simple statistic and print it out.
# We want to verify that the task of chunking will produce the same results!

nMale=0 # number of male records
for i in range(2571):
    if sDF.iloc[i, -5]=="m":
        nMale+=1
nFemale=2571-nMale #number of female records

print('Number male=',nMale, '\nNumber female=',nFemale)

Number male= 1243 
Number female= 1328


<h3>Chunking</h3>One of downsides of dataFrames is that the entire data set needs to reside in memory.  In the context of big data, that cannot be reasonable.  What one can do is process the data in chunks.  So earlier in pure Python we processed files line by line, which does not limited you by file size (advantage of the previous assignment). Here we can do a iterate by reading smaller blocks of lines (chunks) and placing them in to a dataFrame.  But, we now have to iterate through however many chunks are necessary to get through the file.  That is, we read a chunk, process/transform, write it out, read the next chunk, etc.
<p>Let's repeat the transformations for the China Suicide data but pretend it is a huge file and we will process it in chunks. For this exercise use chunks of 100, for the experience of writing chunking code.  Normally you try to have the fewest number of chunks.

In [127]:
import pandas as pd

# set the file to your China Suicide data file
data_iterator = pd.read_csv("/Users/Charley/Dropbox/Charley/Juniata/Data Science Masters/Data Acquisition & Visualization/SuicideChina.csv", chunksize=100)

# counters
nMale = 0
nFemale = 0
# Each chunk is in dataframe format
headerFlag = True # first time output headers with chunk
for data_chunk in data_iterator:  
    
     # apply the transformations to the data_chunk

    # eliminate the first two id columns
    del data_chunk['Person_ID']


    drop2 = data_chunk.columns[data_chunk.columns.str.contains('Unnamed')]

    data_chunk.drop(drop2, axis=1, inplace=True)
    
    # The columns of Hospitalised, Died, Urban all have data coded as yes/no.  
    #   Change the data to 1 or 0.  Anything not yes/no should be coded as NA.
    for i in range(3):
        for k in range(len(data_chunk)):
            if data_chunk.iloc[k, i]=="yes":
                data_chunk.iloc[k, i]=1
            elif data_chunk.iloc[k, i]=="no":
                data_chunk.iloc[k, i]=0
            else:
                data_chunk.iloc[k, i]="NA"
    dsex=data_chunk["Sex"]
    ndsex=dsex.apply(accessCharacter)
    data_chunk["Sex"]=ndsex
    dedu=data_chunk["Education"]
    ndedu=dedu.apply(fixUp)
    data_chunk["Education"]=ndedu
    data_chunk = data_chunk.rename(columns={'method': 'Method'})
    dmet=data_chunk["Method"]
    ndmet=dmet.apply(fixUp2)
    data_chunk["Method"]=ndmet
    docc=data_chunk["Occupation"]
    ndocc=docc.apply(fixUp)
    data_chunk["Occupation"]=ndocc
    mMale=0
    for i in range(len(data_chunk)):
        if data_chunk.iloc[i, -5]=="m":
            mMale+=1
    mFemale=len(data_chunk)-mMale
    
    # write out the chunk; first time with headers
    data_chunk.to_csv("/Users/Charley/Dropbox/Charley/Juniata/Data Science Masters/Data Acquisition & Visualization/SuicideChinaTransfChunked.csv", mode='a', header=headerFlag) 
        #you need to append, default is write otherwise you only keep the last chunk
    headerFlag=False # no more headers

    nMale += mMale #number of males in this chunk
    nFemale += mFemale #number of females in this chunk
    
print('Number male=',nMale, '\nNumber female=',nFemale)

Number male= 1243 
Number female= 1328
